# Demo 07d — Stacking and the Choice of a Model

**Class 07 · E5, E5b** (slides: `slides/07_ensembles.md`)

Voting gives every member a fixed say; **stacking learns how much to trust each member**. Then the question of the class: on real tables, which method do we use?

* **E5 — Stacking:** a logistic-regression meta-learner on the predicted probabilities of 1-NN, a tree and an SVM, on the moons. Three ways to build its training features: **in-fold** (the members predict the rows they were trained on: a leak), **out-of-fold** (5-fold cross-validated predictions: the right way) and **blending** (a hold-out). The leak measured as an optimistic score; a sweep of the meta-learner strength; ten seeds.
* **E5b — Real data:** ten models and three combinations on heart disease, breast cancer (classification) and a subsample of California housing (regression): repeated 5-fold cross-validation with the same folds for every model, the corrected resampled t-test of Class 01, fit and predict time; the reliability of the probabilities of a forest, boosting and logistic regression on spambase.

Convention (Class 01): class 1 is the positive class: disease for heart disease, **malignant** for breast cancer (the file codes benign as 1: recoded), spam for spambase. Every cell that prints a number of a slide starts with `slide: <frame title>`.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from joblib import Parallel, delayed
from matplotlib.ticker import FormatStrFormatter, MaxNLocator
from scipy import stats
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.datasets import make_moons
from sklearn.ensemble import (
    AdaBoostClassifier,
    AdaBoostRegressor,
    ExtraTreesClassifier,
    ExtraTreesRegressor,
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
    StackingClassifier,
    StackingRegressor,
    VotingClassifier,
    VotingRegressor,
)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.model_selection import (
    RepeatedKFold,
    RepeatedStratifiedKFold,
    StratifiedKFold,
    cross_val_predict,
    cross_val_score,
    train_test_split,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from threadpoolctl import threadpool_limits

from faa import boundaries, data, galleries, metrics, plotlib

warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(20)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
pl.Config.set_fmt_str_lengths(70)


plotlib.setup(module="07")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
SEED = 0
N_JOBS = 8


def slide(title: str) -> None:
    """Every cell that prints a number of a slide starts with this line: the frame title of the deck."""
    print(f"--- slide: {title} ---")


def show(frame: pl.DataFrame, digits: int = 3) -> None:
    """Print a polars table with rounded floats."""
    print(frame.with_columns(pl.col(pl.Float64).round(digits)))


moons_X, moons_y = make_moons(300, noise=0.3, random_state=SEED)  # the running example of the deck
X_tr, X_te, y_tr, y_te = data.split(moons_X, moons_y, SEED)  # 210 train, 90 test
LIM = boundaries.plane_limits(moons_X)

## E5 — Stacking on the moons

Members: **1-NN, an unpruned tree and an RBF SVM**. The first two are perfect on the rows they were trained on (train accuracy 1.000), which is what makes the leak visible. The meta-learner is a logistic regression on the three predicted probabilities. `HandStack` has three modes:

* **out-of-fold:** each member predicts, for every training row, with a model fitted on the other four folds; the meta-learner trains on those honest predictions; then the members are refitted on all the rows for the test set. `StackingClassifier` does exactly this (cv=5): same predictions as our hand-made version here (0.922, log loss 0.259).
* **in-fold (leak):** the members are fitted on all rows and predict the same rows; 1-NN and the tree say "100 % sure and right", so the meta-learner learns to trust them: weights **3.02, 3.02, 1.57** on [1-NN, tree, SVM] against **1.12, 1.82, 2.62** out-of-fold (where the SVM, the member that does not memorise, gets the largest weight).
* **blending:** the members are fitted on 70 % of the rows, the meta-learner on the other 30 %: no leak, but neither stage sees all the data (0.911 here).

On this split the three stacks reach 0.922, 0.922 and 0.911 against 0.944, 0.878 and 0.889 for the members: the stack is above two of the three and below 1-NN, which is lucky on these 90 test points.

In [ ]:
slide("Stacking: a Meta-Learner on Out-of-Fold Predictions")


class HandStack:
    """Stacking of probability outputs with a logistic-regression meta-learner.
    mode "oof":   meta features = 5-fold out-of-fold probabilities of every member (the right way);
    mode "leak":  meta features = the probabilities of members fitted on ALL training rows, predicted on the same rows;
    mode "blend": members are fitted on 70 % of the rows and the meta-learner on the other 30 % (a hold-out)."""

    def __init__(self, members, mode: str = "oof", seed: int = 0):
        self.members, self.mode, self.seed = members, mode, seed

    def _features(self, models, X):
        return np.column_stack([m.predict_proba(X)[:, 1] for m in models])

    def fit(self, X, y):
        if self.mode == "blend":
            a, b, c, d = train_test_split(X, y, test_size=0.3, stratify=y, random_state=self.seed)
            self.fitted_ = [clone(m).fit(a, c) for _, m in self.members]
            Z, t = self._features(self.fitted_, b), d
        else:
            self.fitted_ = [clone(m).fit(X, y) for _, m in self.members]
            if self.mode == "leak":
                Z = self._features(self.fitted_, X)
            else:
                cv = StratifiedKFold(5, shuffle=True, random_state=self.seed)
                Z = np.column_stack(
                    [cross_val_predict(clone(m), X, y, cv=cv, method="predict_proba")[:, 1] for _, m in self.members]
                )
            t = y
        self.meta_ = LogisticRegression().fit(Z, t)
        self.meta_features_, self.meta_target_ = Z, t
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X):
        return self.meta_.predict_proba(self._features(self.fitted_, X))

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > 0.5).astype(int)

    def score(self, X, y):
        return float(np.mean(self.predict(X) == y))


g = galleries.gallery_models(SEED)
MEMBERS = [("1-NN", g["1-NN"]), ("tree", g["tree"]), ("SVM RBF", g["SVM RBF"])]
stacks = {mode: HandStack(MEMBERS, mode, SEED).fit(X_tr, y_tr) for mode in ("leak", "oof", "blend")}
members = {n: clone(m).fit(X_tr, y_tr) for n, m in MEMBERS}
sk_stack = StackingClassifier(
    [(n, clone(m)) for n, m in MEMBERS],
    final_estimator=LogisticRegression(),
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
).fit(X_tr, y_tr)
rows = [(n, float(m.score(X_tr, y_tr)), *metrics.scores(m, X_te, y_te)) for n, m in members.items()]
rows += [(f"stack ({mode})", float(s.score(X_tr, y_tr)), *metrics.scores(s, X_te, y_te)) for mode, s in stacks.items()]
rows.append(
    ("StackingClassifier (scikit-learn)", float(sk_stack.score(X_tr, y_tr)), *metrics.scores(sk_stack, X_te, y_te))
)
show(pl.DataFrame(rows, schema=["model", "train acc", "test acc", "test log loss"], orient="row"))
for mode in ("leak", "oof", "blend"):
    print(
        f"meta-learner weights ({mode}) on [1-NN, tree, SVM]: {np.round(stacks[mode].meta_.coef_[0], 2).tolist()}, intercept {stacks[mode].meta_.intercept_[0]:+.2f}"
    )

## The leak is an optimistic score

The meta-learner trained on in-fold features is **perfect on its own training rows** (1.000) and its 5-fold CV accuracy on those features is also **1.000**, while the stack has 0.922 on the test set: the estimate is 0.078 too high. With out-of-fold features the same numbers are honest: 0.910 (train), 0.914 (CV), 0.922 (test). The leak is in the **features** of the second stage, not in the model: any score computed on in-fold predictions of overfitting members repeats the pitfall of Class 01 (evaluating on training data).

In [ ]:
slide("The Leak: an Optimistic Score")
rows = []
for mode in ("leak", "oof"):
    Z, t = stacks[mode].meta_features_, stacks[mode].meta_target_
    cv_meta = float(
        cross_val_score(LogisticRegression(), Z, t, cv=StratifiedKFold(5, shuffle=True, random_state=SEED)).mean()
    )
    rows.append((mode, float(stacks[mode].meta_.score(Z, t)), cv_meta, float(stacks[mode].score(X_te, y_te))))
show(
    pl.DataFrame(
        rows,
        schema=[
            "meta features",
            "meta-learner train acc",
            "5-fold CV acc of the meta-learner",
            "test acc of the stack",
        ],
        orient="row",
    )
)
print(
    "with in-fold features the CV score of the meta-learner is near 1.00 (every member is perfect on its own training rows, so the meta-learner learns to trust 1-NN and the tree);"
)
print(
    "with out-of-fold features the same score is honest (about the test accuracy). The leak is in the features, not in the model."
)

In [ ]:
# plot e5_stack
fg, axs = boundaries.new_fig(2, 3, panel_h=1.0)
panels = [(n, m) for n, m in members.items()] + [
    ("stack, in-fold (leak)", stacks["leak"]),
    ("stack, out-of-fold", stacks["oof"]),
    ("stack, blending", stacks["blend"]),
]
for ax, (name, m) in zip(axs.ravel(), panels):
    boundaries.boundary_panel(ax, m, X_tr, y_tr, X_te, y_te, title=name, lim=LIM)
fg.subplots_adjust(wspace=0.06, hspace=0.42)
plotlib.save(fg, "e5_stack")

## Does the leak cost accuracy? A sweep (no slide)

The leak is large in the *score*, smaller in the *test accuracy*, and it depends on how freely the meta-learner can follow it. The table varies the regularisation $C$ of the logistic regression (small $C$ = strong shrinkage) and the noise of the moons, 10 seeds each:

* With strong shrinkage ($C \le 0.1$) the meta-learner cannot over-trust any member and the leak costs nothing (0.000 and 0.001).
* With the default $C = 1$ the out-of-fold stack is 0.008 better (noise 0.3) and 0.017 better (noise 0.4); with a weakly regularised meta-learner ($C = 100$ or more) it is **0.021 to 0.053 better**: at noise 0.4 and $C = 10^4$ the leaky stack drops to 0.804 against 0.858.
* Over ten seeds at $C = 1$: the out-of-fold stack (0.888) matches its best member (SVM, 0.887) and is above 1-NN (0.869) and the tree (0.863); the leaky stack is at 0.880, blending at 0.883. The stack is **not** better than the best member here: the moons are near their ceiling (Demo 07a).

In [ ]:
slide("The Leak Against the Strength of the Meta-Learner (no slide)")


def leak_run(seed, noise):
    X_, y_ = make_moons(300, noise=noise, random_state=seed)
    a, b, c, d = data.split(X_, y_, seed)
    gg = galleries.gallery_models(seed)
    mem = [("1-NN", gg["1-NN"]), ("tree", gg["tree"]), ("SVM RBF", gg["SVM RBF"])]
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    with threadpool_limits(1):
        fitted = [clone(m).fit(a, c) for _, m in mem]
        in_fold = np.column_stack([m.predict_proba(a)[:, 1] for m in fitted])
        out_fold = np.column_stack(
            [cross_val_predict(clone(m), a, c, cv=cv, method="predict_proba")[:, 1] for _, m in mem]
        )
        test = np.column_stack([m.predict_proba(b)[:, 1] for m in fitted])
    rows = []
    for C in (0.01, 0.1, 1.0, 100.0, 10_000.0):
        rows.append((noise, C, "leak", seed, float(LogisticRegression(C=C).fit(in_fold, c).score(test, d))))
        rows.append((noise, C, "out-of-fold", seed, float(LogisticRegression(C=C).fit(out_fold, c).score(test, d))))
    return rows


lr_ = Parallel(n_jobs=N_JOBS)(delayed(leak_run)(s, nz) for s in range(10) for nz in (0.3, 0.4))
ll = pl.DataFrame(
    [r for run in lr_ for r in run], schema=["noise", "C of the meta-learner", "features", "seed", "acc"], orient="row"
)
lw = (
    ll.group_by("noise", "C of the meta-learner", "features")
    .agg(pl.col("acc").mean())
    .pivot(on="features", index=["noise", "C of the meta-learner"], values="acc")
    .sort("noise", "C of the meta-learner")
)
lw = lw.with_columns((pl.col("out-of-fold") - pl.col("leak")).alias("gain of out-of-fold"))
print(
    "test accuracy of the stack (1-NN, tree, SVM) with the meta-learner trained on in-fold (leak) or out-of-fold features, mean of 10 seeds:"
)
show(lw)

In [ ]:
slide("Stacking: Ten Seeds")


def stack_run(seed):
    X_, y_ = make_moons(300, noise=0.3, random_state=seed)
    a, b, c, d = data.split(X_, y_, seed)
    gg = galleries.gallery_models(seed)
    mem = [("1-NN", gg["1-NN"]), ("tree", gg["tree"]), ("SVM RBF", gg["SVM RBF"])]
    with threadpool_limits(1):
        out = [(n, seed, float(clone(m).fit(a, c).score(b, d))) for n, m in mem]
        out += [
            (f"stack ({mode})", seed, float(HandStack(mem, mode, seed).fit(a, c).score(b, d)))
            for mode in ("leak", "oof", "blend")
        ]
        sk = StackingClassifier(
            [(n, clone(m)) for n, m in mem],
            final_estimator=LogisticRegression(),
            cv=StratifiedKFold(5, shuffle=True, random_state=seed),
        )
        out.append(("StackingClassifier", seed, float(sk.fit(a, c).score(b, d))))
    return out


sr = Parallel(n_jobs=N_JOBS)(delayed(stack_run)(s) for s in range(10))
sl = pl.DataFrame([r for run in sr for r in run], schema=["model", "seed", "acc"], orient="row")
order = ["1-NN", "tree", "SVM RBF", "stack (leak)", "stack (oof)", "stack (blend)", "StackingClassifier"]
sa = (
    sl.group_by("model")
    .agg(pl.col("acc").mean(), pl.col("acc").std().alias("sd"))
    .join(pl.DataFrame({"model": order, "i": range(7)}), on="model")
    .sort("i")
    .drop("i")
)
print("moons, noise 0.3, 10 seeds, test accuracy:")
show(sa)
lk = sl.filter(pl.col("model") == "stack (leak)")["acc"].to_numpy()
oo = sl.filter(pl.col("model") == "stack (oof)")["acc"].to_numpy()
print(
    f"out-of-fold stack minus leaky stack: mean {np.mean(oo - lk):+.3f}; better in {int(np.sum(oo > lk))} of 10 seeds, equal in {int(np.sum(oo == lk))}"
)

## E5b — Which method on real tables?

Ten models and three combinations of them, the **same 10 folds for every model** (5-fold cross-validation repeated twice, $J = 10$ scores, $n_{test}/n_{train} = 1/4$). The data: heart disease (303 patients, 26 features after one-hot), breast cancer (569 patients, 30 features) and 2000 districts drawn from California housing (8 features; the table reports RMSE in 100 000 USD, lower is better). The names are common to the two tasks: **linear** = logistic regression or ridge, **SVM** = RBF SVC or SVR; the ensembles are random forest, extra trees, AdaBoost (stumps for classification, trees of depth 4 for regression), `GradientBoosting` and `HistGB`; the three **combinations** are a vote of linear + 15-NN + SVM, a vote of SVM + forest + HistGB, and a stack (ridge or logistic meta-learner, 3-fold) of SVM, forest, HistGB and 15-NN. "p vs best" is the corrected resampled t-test of Class 01 (Nadeau and Bengio) against the best model of the data set: a large p means that the data cannot tell that model from the best. Fit and predict times are the mean per fold in this run, with all folds running in parallel on one machine: **typical, not exact**.

In [ ]:
slide("Real Data: Ten Models and Three Combinations, Three Data Sets")
Xhd, yhd = data.load_heart()
Xbc, ybc = data.load_breast()
Xca, yca = data.load_california(2000, seed=SEED)
print(
    f"heart disease: {Xhd.shape[0]} patients, {Xhd.shape[1]} features after one-hot, {int(yhd.sum())} with disease (y = 1)"
)
print(f"breast cancer: {Xbc.shape[0]} patients, {Xbc.shape[1]} features, {int(ybc.sum())} malignant (y = 1)")
print(f"california housing: {Xca.shape[0]} districts drawn at random, {Xca.shape[1]} features, target in 100 000 USD")


def scaled(m):
    return make_pipeline(StandardScaler(), m)


def svm_p():
    """An SVM with probabilities (Platt scaling inside 5-fold CV), the replacement of SVC(probability=True)."""
    return scaled(CalibratedClassifierCV(SVC(), method="sigmoid", cv=5, ensemble=False))


VOTE_A, VOTE_B, STACK = (
    "vote: linear + 15-NN + SVM",
    "vote: SVM + forest + HistGB",
    "stack: SVM + forest + HistGB + 15-NN",
)


def class_models(seed=0):
    """Ten models and three combinations for a classification table (one-hot or numeric features).
    "linear" is logistic regression, "SVM" is an RBF SVM."""
    lr, knn, svm = (
        scaled(LogisticRegression(max_iter=2000)),
        scaled(KNeighborsClassifier(n_neighbors=15)),
        scaled(SVC()),
    )
    rf = RandomForestClassifier(n_estimators=200, random_state=seed)
    hgb = HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, random_state=seed)
    single = {
        "linear": lr,
        "15-NN": knn,
        "SVM": svm,
        "MLP": scaled(MLPClassifier(hidden_layer_sizes=(32,), alpha=1e-2, max_iter=2000, random_state=seed)),
        "tree": DecisionTreeClassifier(max_depth=5, random_state=seed),
        "random forest": rf,
        "extra trees": ExtraTreesClassifier(n_estimators=200, random_state=seed),
        "AdaBoost": AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1), n_estimators=200, random_state=seed
        ),
        "GradientBoosting": GradientBoostingClassifier(
            n_estimators=100, learning_rate=0.1, max_depth=3, random_state=seed
        ),
        "HistGB": hgb,
    }
    combos = {
        VOTE_A: VotingClassifier([("a", clone(lr)), ("b", clone(knn)), ("c", svm_p())], voting="soft"),
        VOTE_B: VotingClassifier([("a", svm_p()), ("b", clone(rf)), ("c", clone(hgb))], voting="soft"),
        STACK: StackingClassifier(
            [("a", svm_p()), ("b", clone(rf)), ("c", clone(hgb)), ("d", clone(knn))],
            final_estimator=LogisticRegression(),
            cv=3,
        ),
    }
    return single, combos


def reg_models_real(seed=0):
    """Ten regressors and three combinations for the regression table. "linear" is ridge regression, "SVM" is an RBF SVR."""
    ridge = scaled(Ridge(alpha=1.0))
    knn, svr = scaled(KNeighborsRegressor(n_neighbors=15)), scaled(SVR(C=3.0))
    rf = RandomForestRegressor(n_estimators=200, max_features=0.5, random_state=seed)
    hgb = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1, random_state=seed)
    single = {
        "linear": ridge,
        "15-NN": knn,
        "SVM": svr,
        "MLP": scaled(MLPRegressor(hidden_layer_sizes=(64,), alpha=1e-2, max_iter=500, random_state=seed)),
        "tree": DecisionTreeRegressor(max_depth=8, random_state=seed),
        "random forest": rf,
        "extra trees": ExtraTreesRegressor(n_estimators=200, max_features=0.5, random_state=seed),
        "AdaBoost": AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=4), n_estimators=100, random_state=seed
        ),
        "GradientBoosting": GradientBoostingRegressor(
            n_estimators=100, learning_rate=0.1, max_depth=3, random_state=seed
        ),
        "HistGB": hgb,
    }
    combos = {
        VOTE_A: VotingRegressor([("a", clone(ridge)), ("b", clone(knn)), ("c", clone(svr))]),
        VOTE_B: VotingRegressor([("a", clone(svr)), ("b", clone(rf)), ("c", clone(hgb))]),
        STACK: StackingRegressor(
            [("a", clone(svr)), ("b", clone(rf)), ("c", clone(hgb)), ("d", clone(knn))],
            final_estimator=Ridge(alpha=1.0),
            cv=3,
        ),
    }
    return single, combos


def one_fold(model, X, y, train, test, classification):
    """Fit on the training rows of one fold: (score, fit seconds, predict seconds); accuracy or RMSE."""
    with threadpool_limits(1), warnings.catch_warnings():
        warnings.simplefilter("ignore")  # worker processes do not inherit the notebook's filter
        m = clone(model)
        t0 = time.perf_counter()
        m.fit(X[train], y[train])
        t1 = time.perf_counter()
        pred = m.predict(X[test])
        t2 = time.perf_counter()
    score = float(np.mean(pred == y[test])) if classification else metrics.rmse(pred, y[test])
    return score, t1 - t0, t2 - t1


def evaluate_all(jobs):
    """Repeated cross-validation of every model of every data set on the same folds, all folds in one parallel run.
    jobs: {data set: (X, y, models, cv, classification)}. Returns {data set: {model: (fold scores, fit s, predict s)}}."""
    tasks, keys = [], []
    for dname, (X, y, models, cv, classification) in jobs.items():
        folds = list(cv.split(X, y if classification else None))
        for mname, m in models.items():
            for tr, te in folds:
                tasks.append(delayed(one_fold)(m, X, y, tr, te, classification))
                keys.append((dname, mname))
    out = Parallel(n_jobs=-1)(tasks)
    res = {}
    for (dname, mname), (sc, ft, pt) in zip(keys, out):
        res.setdefault(dname, {}).setdefault(mname, []).append((sc, ft, pt))
    return {
        d: {
            m: (np.array([r[0] for r in v]), float(np.mean([r[1] for r in v])), float(np.mean([r[2] for r in v])))
            for m, v in ms.items()
        }
        for d, ms in res.items()
    }

In [ ]:
slide("Real Data: Accuracy, Fit Time and the Corrected Test")
CV_CLASS = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=SEED)  # J = 10 scores, n_test / n_train = 1/4
CV_REG = RepeatedKFold(n_splits=5, n_repeats=2, random_state=SEED)  # J = 10 scores
cs, cc = class_models(SEED)
rs, rc = reg_models_real(SEED)
results = evaluate_all(
    {
        "heart disease": (Xhd, yhd, {**cs, **cc}, CV_CLASS, True),
        "breast cancer": (Xbc, ybc, {**cs, **cc}, CV_CLASS, True),
        "california housing": (Xca, yca, {**rs, **rc}, CV_REG, False),
    }
)
METRIC = {"heart disease": "accuracy", "breast cancer": "accuracy", "california housing": "RMSE"}
J = {"heart disease": 10, "breast cancer": 10, "california housing": 10}


def corrected_p(a, b, ratio):
    """Corrected resampled t-test of Class 01 (Nadeau and Bengio) on the paired fold scores a - b."""
    d = np.asarray(a) - np.asarray(b)
    if d.std(ddof=1) == 0:
        return 1.0
    t = d.mean() / np.sqrt(d.var(ddof=1) * (1 / len(d) + ratio))
    return float(2 * stats.t.sf(abs(t), len(d) - 1))


tables = {}
for dname, res in results.items():
    higher = METRIC[dname] == "accuracy"
    means = {m: float(v[0].mean()) for m, v in res.items()}
    best = max(means, key=means.get) if higher else min(means, key=means.get)
    ratio = 1 / 4
    rows = [
        (m, means[m], float(v[0].std(ddof=1)), corrected_p(v[0], res[best][0], ratio), v[1], v[2])
        for m, v in res.items()
    ]
    tab = pl.DataFrame(
        rows, schema=["model", METRIC[dname], "sd", "p vs best", "fit s", "predict s"], orient="row"
    ).sort(METRIC[dname], descending=higher)
    tables[dname] = (tab, best)
    print(
        f"{dname}: {METRIC[dname]}, mean of {J[dname]} fold scores (5-fold, {J[dname] // 5} repeats), sd over folds, corrected-t p-value against the best model ({best}), mean fit and predict seconds per fold"
    )
    print(
        tab.with_columns(pl.col("p vs best").map_elements(lambda p: f"{p:.2f}", return_dtype=pl.String)).with_columns(
            pl.col(pl.Float64).exclude("p vs best").round(3)
        )
    )

## Reading the real-data tables

* **Heart disease:** the best model is the vote of linear + 15-NN + SVM (0.837), but plain **logistic regression is at 0.835 (p = 0.91)** and no ensemble is better than it: random forest 0.827, HistGB 0.809. The single tree is last (0.761); a forest is 0.066 better than a tree (corrected p = 0.045), HistGB 0.048 better (p = 0.257): with 303 patients the test cannot separate most of the models.
* **Breast cancer:** everything but the tree (0.933) is between 0.963 and 0.979; the best is the vote (0.979), logistic regression and SVM are at 0.975 and 0.974. The forest is 0.030 above the tree (p = 0.093).
* **California housing** is the case where the ensembles earn their cost. RMSE of the stack 0.530, vote of SVM + forest + HistGB 0.533 (p = 0.42 against the stack), **HistGB 0.543**, extra trees 0.555, forest 0.566; a single tree is at 0.746 and ridge regression at 0.661. HistGB is 0.203 better than a tree and 0.023 better than a forest (p = 0.032); the stack is 0.013 better than HistGB (p = 0.001) and fits about **10 times slower** per fold (a typical 7 s against 0.7 s), so on a large table HistGB is the reasonable stopping point.
* **Rule of the slides:** on small, almost linear tables a regularised linear model is a baseline that ensembles rarely beat; on tables with non-linear structure and interactions, boosted trees (and forests) win by a clear margin, and a stack buys a little more at a multiple of the cost. The comparison needs the Class 01 protocol: same folds, a corrected test, a baseline.

In [ ]:
slide("Named Pairs: Does the Ensemble Beat Its Single Model?")
pairs = [
    ("random forest", "tree"),
    ("HistGB", "random forest"),
    ("HistGB", "tree"),
    (STACK, "HistGB"),
    (VOTE_A, "linear"),
]
for dname, res in results.items():
    sign = 1 if METRIC[dname] == "accuracy" else -1
    rows = []
    for a_, b_ in pairs:
        ka, kb = a_, b_
        d = sign * (res[ka][0] - res[kb][0])  # positive = first model better
        rows.append((f"{ka} - {kb}", float(d.mean()), corrected_p(res[ka][0], res[kb][0], 1 / 4)))
    print(f"{dname} ({METRIC[dname]}; positive = the first model is better):")
    print(
        pl.DataFrame(rows, schema=["pair", "mean difference", "corrected p"], orient="row").with_columns(
            pl.col("mean difference").round(4),
            pl.col("corrected p").map_elements(lambda p: f"{p:.3f}", return_dtype=pl.String),
        )
    )

In [ ]:
# plot e5b_scores: mean score and sd over the folds, one panel per data set, one row order for the three
FAMILY = {
    "tree": GREY,
    "random forest": ORANGE,
    "extra trees": ORANGE,
    "AdaBoost": RED,
    "GradientBoosting": RED,
    "HistGB": RED,
}
rank_tab = pl.concat(
    [
        t.with_columns(pl.col(METRIC[d]).rank(descending=METRIC[d] == "accuracy").alias("rank")).select("model", "rank")
        for d, (t, _) in tables.items()
    ]
)
names = rank_tab.group_by("model").agg(pl.col("rank").mean()).sort("rank")["model"].to_list()  # best mean rank first
ypos = np.arange(len(names))
cols = [GREEN if n.startswith(("vote", "stack")) else FAMILY.get(n, BLUE) for n in names]
fg, axs = plt.subplots(1, 3, figsize=(plotlib.FULL, 2.3), sharey=True)
for ax, (dname, (tab, best)) in zip(axs, tables.items()):
    t = tab.to_dict(as_series=False)
    idx = {m: i for i, m in enumerate(t["model"])}
    x = np.array([t[METRIC[dname]][idx[n]] for n in names])
    sd = np.array([t["sd"][idx[n]] for n in names])
    ax.errorbar(x, ypos, xerr=sd, fmt="none", ecolor="#999999", elinewidth=0.6, capsize=0)
    ax.scatter(x, ypos, c=cols, s=14, zorder=3)
    ax.xaxis.set_major_locator(MaxNLocator(3))
    ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    ax.tick_params(axis="x", labelsize=6.3)
    ax.set_xlabel("accuracy" if METRIC[dname] == "accuracy" else "RMSE (lower = better)", fontsize=6.5, labelpad=1.5)
    ax.set_title(dname, fontsize=7, pad=2)
    ax.grid(axis="y", visible=False)
axs[0].set_yticks(ypos, names, fontsize=5.6)
axs[0].set_ylim(len(names) - 0.5, -0.5)  # best mean rank at the top
fg.subplots_adjust(wspace=0.2, left=0.36)
plotlib.save(fg, "e5b_scores")

## Calibration: can the probabilities be trusted?

Out-of-fold probabilities of 5-fold CV on spambase (4601 e-mails, 1813 spam). The **reliability curve** plots, in 10 equal-count bins of predicted probability, the observed share of spam; the diagonal is a calibrated model. The **calibration gap** is the mean distance to the diagonal.

* **HistGB** (log-loss objective) has the smallest gap, 0.024 (log loss 0.144), **logistic regression** 0.032 (0.219; it is calibrated by construction, but less accurate: 0.924), the **random forest** 0.045 (0.151): a forest averages trees that vote 0 or 1, so its probabilities rarely reach the ends of the scale.
* **AdaBoost** has a gap of **0.354** (log loss 0.549): its scores live between about 0.2 and 0.75, an S-shaped curve; the histogram shows that it never says "99 %". Accurate in its decisions (0.939), poor as a probability. **Naive Bayes** is the opposite: over-confident (gap 0.206, log loss 1.202, accuracy 0.821).
* A model whose probabilities feed a cost-based decision (Class 04) may need calibration (Platt scaling, isotonic) on top of the ensemble.

In [ ]:
slide("Calibration: Reliability of Probabilities (Spambase)")
Xsp, ysp = data.load_spam()
cal_models = {
    "logistic": scaled(LogisticRegression(max_iter=3000)),
    "naive Bayes": GaussianNB(),
    "random forest": RandomForestClassifier(n_estimators=200, random_state=SEED),
    "AdaBoost": AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=200, random_state=SEED),
    "HistGB": HistGradientBoostingClassifier(max_iter=200, random_state=SEED),
}
cv5 = StratifiedKFold(5, shuffle=True, random_state=SEED)
with threadpool_limits(1):
    probs = dict(
        zip(
            cal_models,
            Parallel(n_jobs=N_JOBS)(
                delayed(cross_val_predict)(m, Xsp, ysp, cv=cv5, method="predict_proba") for m in cal_models.values()
            ),
        )
    )


def ece(y, p, bins=10):
    """Expected calibration error: weighted mean gap between the predicted probability and the observed share, 10 equal-count bins."""
    frac, mean_p = calibration_curve(y, p, n_bins=bins, strategy="quantile")
    return float(np.mean(np.abs(frac - mean_p)))


rows = [
    (
        n,
        float(np.mean((p[:, 1] > 0.5) == ysp)),
        float(log_loss(ysp, np.clip(p[:, 1], 1e-3, 1 - 1e-3))),
        float(brier_score_loss(ysp, p[:, 1])),
        ece(ysp, p[:, 1]),
    )
    for n, p in probs.items()
]
cal = pl.DataFrame(
    rows, schema=["model", "accuracy", "log loss", "Brier score", "calibration gap (10 bins)"], orient="row"
)
show(cal)
print(
    f"spambase, {len(ysp)} e-mails, {int(ysp.sum())} spam; out-of-fold probabilities of 5-fold CV (every e-mail is predicted by a model that did not see it)"
)

In [ ]:
# plot e5b_reliability
fg, axs = plt.subplots(1, 2, figsize=(plotlib.FULL, 1.7))
ax = axs[0]
ax.plot([0, 1], [0, 1], color="k", lw=0.6, ls=":")
for (name, p), col in zip(probs.items(), (BLUE, GREY, ORANGE, RED, GREEN)):
    frac, mean_p = calibration_curve(ysp, p[:, 1], n_bins=10, strategy="quantile")
    ax.plot(mean_p, frac, color=col, marker="o", ms=2.3, label=name)
ax.set(xlabel="predicted probability of spam", ylabel="observed share of spam", xlim=(0, 1), ylim=(0, 1))
ax.legend(loc="upper left", fontsize=6.0, handlelength=1.4, labelspacing=0.25, borderaxespad=0.2)
ax = axs[1]
for (name, p), col in zip(probs.items(), (BLUE, GREY, ORANGE, RED, GREEN)):
    if name in ("logistic", "random forest", "AdaBoost", "HistGB"):
        ax.hist(p[:, 1], bins=np.linspace(0, 1, 21), histtype="step", color=col, lw=1.1, label=name)
ax.set(xlabel="predicted probability of spam", ylabel="e-mails (log scale)", yscale="log", ylim=(0.7, 3e5))
ax.legend(loc="upper center", ncols=2, fontsize=6.0, handlelength=1.4, labelspacing=0.25, columnspacing=0.8)
fg.subplots_adjust(wspace=0.32)
plotlib.save(fg, "e5b_reliability")

## Summary

* Stacking trains a meta-learner on **out-of-fold** predictions; with in-fold predictions the meta-learner trusts the members that memorise, and its score is optimistic (1.000 against 0.922 on the moons).
* A vote or a stack of members of similar quality is as good as the best member and not worse; it costs training and prediction time, and the stack costs the most.
* Real tables: linear baselines win or tie on small, linear data (heart disease, breast cancer); boosted trees win on non-linear data (California); the corrected t-test says which differences are not noise.
* Probabilities: HistGB and logistic regression are well calibrated, a forest slightly under-confident, AdaBoost far from calibrated.
* Next: the lab (bagging, AdaBoost and gradient boosting written by hand and compared with their scikit-learn twins).